In [ ]:
EXPECTED_PARROT_API_KEY = "api_key"

In [ ]:
import os
import pandas as pd
import concurrent.futures
from datetime import datetime
from edsl import QuestionNumerical, Scenario, ScenarioList, Model, ModelList, Survey, AgentList

# Create a directory to store the results
results_dir = "fed_treatment_results"
os.makedirs(results_dir, exist_ok=True)

# Create questions about Federal Reserve policy
q3 = QuestionNumerical(
    question_name = "Q1_S",
    question_text = """
    Consider the following current event: {{ info }}
    What do you expect the rate of inflation to be over the next 12 months? Please give your best guess. 
    """,
    # min_value = 1
    # max_value = 100 
)

q4 = QuestionNumerical(
    question_name = "Q2_L",
    question_text = """
    Consider the following current event: {{ info }}
    What do you expect the rate of inflation to be over the 12-month period beginning 24 months from now
    and ending 36 months from now? 
    Please give your best guess.
    """,
    # min_value = 1, 
    # max_value = 100 
)

# Create scenarios from your treatments
treatments = [
    # Control (no information)
    {
        "treatment": "T_0",
        "description": "Control with no information",
        "info": ""
    },
    # a. Changes in Language Complexity
    # Neutral Information
    {
        "treatment": "T_a1",
        "description": "Neutral - Simplified language",
        "info": (
            "The Federal Reserve monitors economic data like employment figures, prices, "
            "and economic growth to make decisions about interest rates. The current "
            "Federal Funds Rate is 4.25%-4.5%."
        )
    },
    {
        "treatment": "T_a2",
        "description": "Neutral - Technical language",
        "info": (
            "The Federal Open Market Committee utilizes a range of economic indicators "
            "including labor market conditions, inflation pressures, inflation expectations, "
            "and financial developments to calibrate monetary policy. The target range "
            "for the federal funds rate is currently 4.25 to 4.50 percent."
        )
    },
    # Add all the other treatments...
    # Anti-Recession (Accommodative Policy)
    {
        "treatment": "T_a3",
        "description": "Anti-recession - Simplified language",
        "info": (
            "The Federal Reserve is lowering interest rates to help boost the economy. "
            "This makes it cheaper for people and businesses to borrow money, which can "
            "create more jobs and economic activity."
        )
    },
    {
        "treatment": "T_a4",
        "description": "Anti-recession - Technical language",
        "info": (
            "The Federal Open Market Committee is implementing accommodative monetary policy "
            "by reducing the target range for the federal funds rate to stimulate aggregate demand. "
            "This policy adjustment is intended to facilitate credit accessibility, promote employment "
            "growth, and support economic expansion."
        )
    },
    # Anti-Inflation (Restrictive Policy)
    {
        "treatment": "T_a5",
        "description": "Anti-inflation - Simplified language",
        "info": (
            "The Federal Reserve is raising interest rates to help bring down high prices. "
            "Higher interest rates make borrowing more expensive, which slows down spending "
            "and helps control rising costs."
        )
    },
    {
        "treatment": "T_a6",
        "description": "Anti-inflation - Technical language",
        "info": (
            "The Federal Open Market Committee is implementing contractionary monetary policy "
            "by increasing the target range for the federal funds rate to counter inflationary pressures. "
            "This policy stance is designed to moderate demand, restore price stability, and anchor "
            "inflation expectations at levels consistent with the Committee's 2 percent objective."
        )
    },
    # b. Framing of Policy Commitments
    # Neutral Information
    {
        "treatment": "T_b1",
        "description": "Neutral - Conditional statement",
        "info": (
            "The Federal Reserve will adjust interest rates based on incoming economic data. "
            "Future policy decisions will depend on developments in employment, inflation, and "
            "broader economic conditions."
        )
    },
    {
        "treatment": "T_b2",
        "description": "Neutral - Unconditional statement",
        "info": (
            "The Federal Reserve will hold its next policy meeting on June 17-18. The committee "
            "will issue its regular statement and economic projections following the conclusion "
            "of the meeting."
        )
    },
    # Remaining treatments...
]

# Function to run a single experiment with a batch of agents
def run_experiment_with_agents(start_index, end_index, run_id):
    try:
        print(f"Running experiment {run_id} with agents from index {start_index} to {end_index}")
        
        # Pull the agent list
        agent_list = AgentList.pull("2c1e88e9-235d-48a3-9050-dc166bce3026")
        
        # Select a subset of agents using the provided indices
        selected_agents = agent_list[start_index:end_index]
        
        print(f"Selected {len(selected_agents)} agents for this run")
        
        # Create scenario list from treatments
        s = ScenarioList(Scenario(treatment) for treatment in treatments)
        
        # Create survey
        survey = Survey(questions=[q3, q4])
        
        # Specify the models
        models = ModelList([
            Model("gpt-4.1-2025-04-14", service_name="openai", temperature=1),
            Model("claude-3-7-sonnet-20250219", service_name="anthropic"),
            Model("meta-llama/Meta-Llama-3-70B-Instruct", service_name="deep_infra"),
        ])
        
        # Run the experiment with the selected agents
        results = survey.by(s).by(selected_agents).by(models).run()
        
        # Generate a timestamp for the filename
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # Create a filename that indicates the agent range
        filename = f"fed_treatment_agents_{start_index}_{end_index}_run_{run_id}_{timestamp}.csv"
        filepath = os.path.join(results_dir, filename)
        
        # Save results to CSV
        results.to_csv(filepath)
        
        print(f"Results for agents {start_index}-{end_index} saved to {filepath}")
        
        return filepath
    except Exception as e:
        print(f"Error in run with agents {start_index}-{end_index}: {str(e)}")
        raise e

# Function to merge all CSV files into one
def merge_all_results(file_paths):
    print("Merging all results...")
    
    # Initialize an empty list to store all dataframes
    all_dfs = []
    
    # Read each CSV file and add a batch_id column
    for i, file_path in enumerate(file_paths):
        try:
            df = pd.read_csv(file_path)
            df['batch_id'] = i + 1  # Add a batch_id column (1-indexed)
            all_dfs.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {str(e)}")
    
    if not all_dfs:
        print("No valid CSV files found to merge.")
        return None
    
    # Concatenate all dataframes
    merged_df = pd.concat(all_dfs, ignore_index=True)
    
    # Save the merged dataframe
    merged_filepath = os.path.join(results_dir, "fed_treatment_all_results.csv")
    merged_df.to_csv(merged_filepath, index=False)
    
    print(f"All results merged and saved to {merged_filepath}")
    
    return merged_filepath

# Main execution
def main():
    # Define how you want to split the 758 agents
    # For example, split into batches of 19 agents each
    batch_size = 10
    total_agents = 758
    
    # Calculate number of batches (rounded up)
    import math
    num_batches = math.ceil(total_agents / batch_size)
    
    print(f"Processing {total_agents} agents in {num_batches} batches of {batch_size} agents each")
    
    file_paths = []
    max_workers = 4  # Adjust based on your system capabilities
    
    # Using ThreadPoolExecutor to run jobs in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all batches to the executor
        future_to_batch = {}
        
        for batch_id in range(num_batches):
            start_index = batch_id * batch_size
            end_index = min((batch_id + 1) * batch_size, total_agents)
            
            future = executor.submit(run_experiment_with_agents, start_index, end_index, batch_id + 1)
            future_to_batch[future] = (start_index, end_index, batch_id + 1)
        
        # Process results as they complete
        for future in concurrent.futures.as_completed(future_to_batch):
            start_index, end_index, batch_id = future_to_batch[future]
            try:
                filepath = future.result()
                file_paths.append(filepath)
                print(f"Batch {batch_id} (agents {start_index}-{end_index}) completed successfully")
            except Exception as e:
                print(f"Batch {batch_id} (agents {start_index}-{end_index}) generated an exception: {e}")
    
    # Sort file_paths by batch number to ensure consistent order
    file_paths.sort()
    
    # Merge all results
    merged_filepath = merge_all_results(file_paths)
    
    print(f"Experiment completed successfully with {num_batches} batches.")
    print(f"Individual results are in the '{results_dir}' folder.")
    if merged_filepath:
        print(f"Combined results are in '{merged_filepath}'.")

if __name__ == "__main__":
    main()